In [1]:
#Import packages 
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import rioxarray
import geopandas as gpd
from pathlib import Path
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colors import LightSource
from matplotlib.colors import ListedColormap
from shapely.geometry import box
import cmocean
import verde as vd
from gstatsMCMC import Topography

ModuleNotFoundError: No module named 'rioxarray'

In [ ]:
# Importing the moa image and trimming
moa = rioxarray.open_rasterio('../../DEMOGORGN data/moa/moa750_2014_hp1_v01.tif')
moa_coast=gpd.read_file('../../DEMOGORGN data/moa/coastlines/moa2014_coastline_v01.shp')
moa_gl=gpd.read_file('../../DEMOGORGN data/moa/coastlines/moa2014_grounding_line_v01.shp')
moa_islands = gpd.read_file('../../DEMOGORGN data/moa/coastlines/moa2014_islands_v01.shp')

squeezed_moa = moa.squeeze('band')

xmin = -1425250
xmax = -1225250
ymin = 80250
ymax = 197250
x_trim = (moa.x > xmin) & (moa.x < xmax)
y_trim = (moa.y > ymin) & (moa.y < ymax)
moa_trim = moa.sel(x=x_trim, y=y_trim, band=1)

In [ ]:
# Getting rid of the 'band' dimension in the moa file and creating a location mask.
# Step 1: Select the first element of the 'band' dimension to eliminate it
moa_dropped = moa.isel(band=0)

# Step 2: Drop the 'band' coordinate if it exists
if 'band' in moa_dropped.coords:
    moa_dropped = moa_dropped.drop_vars('band')

print(moa_dropped.dims)

X,Y=np.meshgrid(moa.x,moa.y)

location_mask=((xmin<X) & (X<xmax)
               & (ymin<Y) & (Y<ymax))

In [ ]:
w=xmax-xmin
h=ymax-ymin
print(w/1000)
print(h/1000)

resolution=500

In [ ]:
xx, yy = np.meshgrid(np.arange(xmin, xmax, resolution), 
                     np.arange(ymin, ymax, resolution))
data = {'x': xx.flatten(),
        'y': yy.flatten()}
df = pd.DataFrame(data)
df

In [ ]:
bedmap_mask, bedmap_surf, bedmap_bed, bedmap_bed_uncertainty, fig = Topography.load_bedmap('../../DEMOGORGN data/bedmap3.nc', 
                                                                                xx, yy, resolution)
fig

In [ ]:
from gstatsMCMC import Topography
print('loading InSAR_MEaSUREs velocity dataset')
velx, vely, velxerr, velyerr, figvel = Topography.load_vel_measures('../../DEMOGORGN data/antarctica_ice_velocity_450m_v2.nc', 
                                                                    xx, yy, resolution)
figvel

In [ ]:
vel_mag=np.sqrt(velx**2+vely**2)

In [ ]:
df = pd.read_csv('RutfordDataGridded.csv')

In [ ]:
# create a grid of x and y coordinates
x_uniq = np.unique(df.x)
y_uniq = np.unique(df.y)

sxmin = np.min(x_uniq)
sxmax = np.max(x_uniq)
symin = np.min(y_uniq)
symax = np.max(y_uniq)

cols = len(x_uniq)
rows = len(y_uniq)

resolution = 500

sxx, syy = np.meshgrid(x_uniq, y_uniq)

In [ ]:
# load other data
dhdt = df['dhdt'].values.reshape(sxx.shape)
smb = df['smb'].values.reshape(sxx.shape)
velx = df['velx'].values.reshape(sxx.shape)
vely = df['vely'].values.reshape(sxx.shape)
bedmap_mask = df['bedmap_mask'].values.reshape(sxx.shape)
bedmachine_thickness = df['bedmachine_thickness'].values.reshape(sxx.shape)
bedmap_surf = df['bedmap_surf'].values.reshape(sxx.shape)
highvel_mask = df['highvel_mask'].values.reshape(sxx.shape)
bedmap_bed = df['bedmap_bed'].values.reshape(sxx.shape)
bedmachine_bed = bedmap_surf - bedmachine_thickness

In [ ]:
#Map Plot

fig, ax = plt.subplots(1, 1, figsize=(12,5))

map2= plt.pcolormesh(xx/1000, yy/1000, vel_mag, cmap='cmo.matter')

cbar = fig.colorbar(map2, ax=ax, label='Velocity (m/yr)')
# cbar_ax = fig.add_axes([0.1, 0.53, 0.02, 0.35])  # [left, bottom, width, height]
# fig.colorbar(map2, cax=cbar_ax, label='M/yr', orientation='horizontal')

# ax_inset = ax.inset_axes([0.65, 0.05, 0.3, 0.3]) # [x, y, width, height] in relative coordinates via percentages
# # # from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# # # ax_inset = inset_axes(
# # #     ax,
# # #     width="35%",
# # #     height="35%",
# # #     loc='lower right',
# # #     borderpad=1
# # # )
# ax_inset.set_facecolor('white')
# moa_coast.plot(facecolor='lightblue', edgecolor='k', ax=ax_inset, linewidth=0.2)
# moa_gl.plot(facecolor='white', edgecolor='k', ax=ax_inset, linewidth=0.2)
# moa_islands.plot(facecolor='white', edgecolor='k', ax=ax_inset, linewidth=0.2)
# ax_inset.contour(moa.x, moa.y,location_mask, levels=[0.8], colors='red', linewidths=2)
# ax_inset.axis('scaled')
# ax_inset.set_xlim([-3e6, 3e6])
# ax_inset.set_ylim([-2.5e6, 2.5e6])
# ax_inset.set_yticks([])  
# ax_inset.set_ylabel('') 
# ax_inset.set_xticks([])  
# ax_inset.set_xlabel('') 


plt.axis('scaled')
ax.set_title('a. Study Area', fontsize=25)
ax.set_xlim([xmin/1000, xmax/1000])
ax.set_ylim([ymin/1000, ymax/1000])
ax.set_xlabel('Polar Sterographic X (km)')
ax.set_ylabel('Polar Sterographic Y (km)')
plt.savefig('Study Area Velocity.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,7))
ax.set_facecolor('white')
moa_coast.plot(facecolor='lightblue', edgecolor='k', ax=ax, linewidth=0.2)
moa_gl.plot(facecolor='white', edgecolor='k', ax=ax, linewidth=0.2)
moa_islands.plot(facecolor='white', edgecolor='k', ax=ax, linewidth=0.2)
ax.contour(moa.x, moa.y,location_mask, levels=[0.8], colors='red', linewidths=2)
ax.axis('scaled')
ax.set_xlim([-3e6, 3e6])
ax.set_ylim([-2.5e6, 2.5e6])
ax_inset.set_ylabel('') 
ax_inset.set_xlabel('') 
#plt.savefig('Map of Antartica.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
norm=plt.Normalize(vmin=-5000, vmax=5000)

fig, ax = plt.subplots(figsize=(12,5))
im=ax.pcolormesh(sxx/1000, syy/1000, bedmap_bed, cmap='cmo.topo', shading='auto', norm=norm)
ax.contour(sxx/1000, syy/1000, highvel_mask, levels=[0], colors='red', alpha=1)
ax.axis('scaled')
ax.set_title('c. Bed Elevation', fontsize=25)
ax.set_xlabel('Polar Sterographic X (km)')
ax.set_ylabel('Polar Sterographic Y (km)')
plt.colorbar(im,ax=ax, label='m', orientation='vertical')

plt.savefig('Bed Elevation.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print(sxx.shape, syy.shape, bedmap_bed.shape)

In [ ]:
cond_bed = np.where(bedmap_mask == 1, df['bed'].values.reshape(xx.shape), bedmap_bed)
            #where grounded ice cond_bed will be df['bed'] and where not cond_bed will be based on bedmap
df['cond_bed'] = cond_bed.flatten()

cond_bed = df['cond_bed'].values.reshape(sxx.shape)
fig, ax = plt.subplots(figsize=(12,5))
im=ax.pcolormesh(sxx/1000,syy/1000,cond_bed,cmap='cmo.topo')
ax.contour(sxx/1000, syy/1000, highvel_mask, levels=[0], colors='red', alpha=1)
ax.axis('scaled')
ax.set_title('c. Bed Elevation Measurements', fontsize=25)
ax.set_xlabel('Polar Sterographic X (km)')
ax.set_ylabel('Polar Sterographic Y (km)')
plt.colorbar(im, ax=ax,label='m')

#plt.savefig('Bed Elevation.png', dpi=300, bbox_inches='tight')
plt.show()